# Dataset management 03: Download and merge a reviewed METASPACE selection

This tutorial turns the filter configuration exported by the METASPACE explorer into a reproducible selection, retrieves the selected imzML/ibd pairs and spatial molecular annotations, and writes one merged dataset.

Interactive dataset discovery is outside this notebook. Complete that workflow first in [Dataset management 02](dataset_management_02_metaspace_explorer.ipynb). This notebook uses `download-merge`, so source pairs are processed one at a time and removed after they are appended.

## 1. Initialize the tutorial environment

The following cell locates the repository root and makes relative paths independent of the directory from which Jupyter was started.

In [ ]:
from pathlib import Path
import os

repository_root = next(
    parent
    for candidate in (Path.cwd(), *Path.cwd().parents)
    for parent in (candidate,)
    if (parent / "pyproject.toml").is_file()
)
os.chdir(repository_root)
repository_root

PosixPath('/home/maxi7524/repositories/MSIAutoEncoderWrapper')

## 2. Resolve the reviewed filters to a selection

The explorer exports filters, not downloadable files. `query` executes those filters once and stores the accepted dataset IDs in a selection JSON. The selection is the reproducible input to download.

The example uses `assets/configs/datasets/metaspace_mouse_liver.json`. Replace this path when the explorer exported another configuration.

In [3]:
%%bash
set -euo pipefail
uv run python assets/scripts/datasets/manage_datasets.py query \
  --workspace-path workspace \
  --source metaspace \
  --filters assets/configs/datasets/metaspace_mouse_liver.json \
  --selection workspace/datasets/selections/metaspace-mouse-liver.json

Repository root: /home/maxi7524/repositories/MSIAutoEncoderWrapper
Workspace path: /home/maxi7524/repositories/MSIAutoEncoderWrapper/workspace
Dataset catalog: /home/maxi7524/repositories/MSIAutoEncoderWrapper/workspace/datasets/catalog.sqlite
2026-08-02 01:11:12,972 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 3 implementation module(s) in package 'msi_autoencoder_wrapper.dataset_management.sources.strategies'.
2026-08-02 01:11:37,264 | INFO     | msi_autoencoder_wrapper.dataset_management.sources.strategies.metaspace:246 | Loaded 19568 METASPACE catalogue records from API
Query filters: /home/maxi7524/repositories/MSIAutoEncoderWrapper/assets/configs/datasets/metaspace_mouse_liver.json
Query selection: /home/maxi7524/repositories/MSIAutoEncoderWrapper/workspace/datasets/selections/metaspace-mouse-liver.json


100%|███████████████████████████████         | 1118 [00:00<00:11,  1.51it/s]/18 [00:00<00:00, 17.84it/s]█████████| 18/18 [00:00<00:00, 21.65it/s]
100%|███████████████████████████████         | 1118 [00:00<00:02,  8.38it/s]/18 [00:00<00:00, 52.01it/s]█████████| 18/18 [00:00<00:00, 55.22it/s]
100%|███████████████████████████████         | 5/13 [00:00<00:00, 13.51it/s]4.40it/s]█████████| 13/13 [00:00<00:00, 31.47it/s]
100%|███████████████████████████████         | 1118 [00:00<00:01,  8.90it/s]/18 [00:00<00:00, 51.00it/s]█████████| 18/18 [00:00<00:00, 53.26it/s]
100%|███████████████████████████████         | 1/14 [00:00<00:01,  7.22it/s]███▎     | 12/14 [00:00<00:00, 39.75it/s]█████████| 14/14 [00:00<00:00, 34.58it/s]


2026-08-02 01:11:42,443 | INFO     | msi_autoencoder_wrapper.dataset_management.sources.strategies.metaspace:215 | METASPACE discovery accepted 2 datasets and rejected 0 datasets
2026-08-02 01:11:42,501 | INFO     | msi_autoencoder_wrapper.dataset_management.operations.query:48 | Wrote query selection with 2 datasets to /home/maxi7524/repositories/MSIAutoEncoderWrapper/workspace/datasets/selections/metaspace-mouse-liver.json


### Review the materialization boundary

Inspect the selected IDs before starting a quota-limited download. The stored `filters.annotation_fdr` will also be used to retrieve molecular results and ion images.

In [4]:
import json

selection_path = Path("workspace/datasets/selections/metaspace-mouse-liver.json")
selection = json.loads(selection_path.read_text(encoding="utf-8"))
print("Source:", selection["source"])
print("annotation_fdr:", selection["filters"].get("annotation_fdr"))
print("Datasets:", len(selection["datasets"]))
[(item["dataset_id"], item["name"]) for item in selection["datasets"]]

Source: metaspace
annotation_fdr: 0.1
Datasets: 2


[('2024-06-12_15h42m44s', 'NEDC_imaging_liver_L1_replicate1'),
 ('2023-10-25_16h42m47s', 'NEDC_imaging_liver')]

## 3. Start an authenticated METASPACE session

Create or sign in to an account at <https://metaspace2020.eu> and generate an API key at <https://metaspace2020.eu/user/me>. 

> METASPACE applies an account or service download quota, its official Python-client documentation does not publish a stable numeric default. The adapter stops before transferring files when the service returns its quota sentinel.

A child shell cannot export a variable back to an already running notebook kernel. Before starting Jupyter, run the following commands in a terminal from the repository root:

```bash
source assets/scripts/datasets/metaspace_session.sh
uv run jupyter lab
```

The script reads the key without echoing it and validates the session through `metaspace_authentication.py`. The notebook inherits `METASPACE_API_KEY` from the shell that starts Jupyter. The next cell verifies that inherited session without displaying the secret.

In [ ]:
%%bash
set -euo pipefail
if [[ -z "${METASPACE_API_KEY:-}" ]]; then
  echo "METASPACE_API_KEY is unavailable. Source assets/scripts/datasets/metaspace_session.sh and restart Jupyter from that shell." >&2
  exit 1
fi
uv run python -m msi_autoencoder_wrapper.dataset_management.sources.strategies.metaspace_authentication

METASPACE_API_KEY is unavailable. Source assets/scripts/datasets/metaspace_session.sh and restart Jupyter from that shell.


CalledProcessError: Command 'b'set -euo pipefail\nif [[ -z "${METASPACE_API_KEY:-}" ]]; then\n  echo "METASPACE_API_KEY is unavailable. Source assets/scripts/datasets/metaspace_session.sh and restart Jupyter from that shell." >&2\n  exit 1\nfi\nuv run python -m msi_autoencoder_wrapper.dataset_management.sources.strategies.metaspace_authentication\n'' returned non-zero exit status 1.

## 4. Download annotations and merge spectra

`download-merge` processes one selected dataset at a time. It retrieves a complete imzML/ibd pair, fetches molecular results and first-isotope ion images at the selection's `annotation_fdr`, writes molecule-to-spectrum links to SQLite, appends selected spectra, and releases staging files.

All annotated spectra are included. `--unannotated-ratio 1.0` additionally requests one randomly sampled spectrum without a molecular link per annotated spectrum, capped by availability. These spectra are controls and are not assumed to be biological background. Remove the option to merge only annotated spectra.

Add `--keep-downloads` when the original source pairs must remain under `workspace/datasets/sources/metaspace`. Without it, only the merged pair and SQLite metadata remain.

In [ ]:
%%bash
set -euo pipefail
uv run python assets/scripts/datasets/manage_datasets.py download-merge \
  --workspace-path workspace \
  --source metaspace \
  --selection workspace/datasets/selections/metaspace-mouse-liver.json \
  --annotation-options assets/configs/datasets/metaspace_annotations.json \
  --output workspace/datasets/merged/metaspace-mouse-liver/dataset.imzML \
  --merged-dataset-id metaspace-mouse-liver \
  --row-width 128 \
  --unannotated-ratio 1.0 \
  --random-seed 0 \
  --keep-downloads  

## 5. Verify merged artifacts and annotation provenance

The final pair contains rearranged rectangular coordinates. SQLite preserves the source dataset and source spectrum ID for every merged spectrum, allowing its molecular annotations and source metadata to be recovered.

In [ ]:
from msi_autoencoder_wrapper.annotations import SQLiteAnnotationReader
from msi_autoencoder_wrapper.readers.strategies.pyimzml_reader import PyImzMLReader

merged_path = Path("workspace/datasets/merged/metaspace-mouse-liver/dataset.imzML")
assert merged_path.is_file()
assert merged_path.with_suffix(".ibd").is_file()

merged_reader = PyImzMLReader(merged_path)
annotation_reader = SQLiteAnnotationReader(
    "workspace/datasets/catalog.sqlite",
    merged_dataset_id="metaspace-mouse-liver",
)

print("Merged spectra:", merged_reader.GetNumberOfSpectra())
print("First spectrum source metadata:", annotation_reader.get_spectrum_metadata(0))
annotation_reader.get_spectrum_annotations(0)

## Result

The workflow produced `workspace/datasets/merged/metaspace-mouse-liver/dataset.imzML`, its `.ibd` companion, and reversible molecular annotation provenance in `workspace/datasets/catalog.sqlite`. Detailed configuration and failure handling are documented in `docs/how-to/dataset-management.md`.